# Phase 2 — Load the actual aging observations

## What we are doing and why

The dataset includes several separate experiments and file formats. For this loading exercise, we will work with the **steady-state measurements** in the ten MATLAB files inside the square-gate aging experiment's `Aging Data` folder. These files are organized under Device 2–5. We retain every file segment, including files named `check`, and record its source file so that we do not silently discard or merge test segments before understanding their meaning.

A MATLAB `.mat` file is not automatically a table. The selected files store nested MATLAB structures; in the actual files, `measurement.steadyState` is a record array and each record contains `date`, `timeEpoch`, and a `timeDomain` structure. We will inspect the field names, then load scalar steady-state values into a pandas DataFrame. Transient waveform arrays and SMU characterization CSVs are different data types and stay out of this first table.

**Why this matters in PHM:** keeping device and source-file identity is essential for interpreting test history and avoiding accidental data leakage. Inspecting missingness, types, and sampling intervals helps distinguish genuine measurements from absent or irregular data before any cleaning or degradation analysis.

**Expected output:** a 68,769-row table with one row per saved steady-state record in this experiment folder, identifying device folder, source MAT file, within-file record index, original date/time, and the measured scalar channels found in the MAT structure. The initial inspection found eight numeric signal channels; the `ambientTemperature` field is entirely missing, and eight records have missing values in the other signal fields. Do not fill or drop them in Phase 2.

This table is an extracted view of the raw MAT observations, not a lossless replacement for the raw data. The cell that saves it writes only to `data/processed/`; it never edits `data/raw/`. We create no features, Health Indicator, failure label, or RUL target.

## 1. Locate and inspect the MATLAB files

`Path.rglob` recursively finds files below a directory. We first locate one `Aging Data` folder and list its MAT files; we do not assume all files in the archive belong to the same experiment.

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent

raw_root = project_root / "data" / "raw"
aging_folders = sorted(raw_root.rglob("Aging Data"))
if len(aging_folders) != 1:
    raise FileNotFoundError(
        f"Expected one 'Aging Data' folder under {raw_root}, found {len(aging_folders)}. "
        "Use the dataset-understanding notebook to identify the intended experiment."
    )

aging_folder = aging_folders[0]
mat_files = sorted(aging_folder.rglob("*.mat"))
if not mat_files:
    raise FileNotFoundError(f"No MATLAB files found under {aging_folder}")

print(f"Aging data folder: {aging_folder.relative_to(raw_root)}")
print(f"MATLAB files: {len(mat_files)}")
for path in mat_files:
    print(f"  {path.relative_to(aging_folder)}")

## 2. Inspect actual MATLAB variables and fields

`scipy.io.loadmat` reads the MAT file. The first call inspects a real file before we build the table. Field names are taken from that file rather than guessed. `squeeze_me=True` removes unnecessary one-element array dimensions; `struct_as_record=False` makes MATLAB structs accessible as Python objects with named fields.

In [ ]:
import numpy as np
from scipy.io import loadmat

mat_contents = loadmat(mat_files[0], squeeze_me=True, struct_as_record=False)
variable_names = [name for name in mat_contents if not name.startswith("__")]
print(f"Example file: {mat_files[0].name}")
print("MATLAB variables:", variable_names)

if "measurement" not in mat_contents:
    raise KeyError(f"No 'measurement' variable in {mat_files[0].name}")
measurement = mat_contents["measurement"]
print("Measurement groups:", measurement._fieldnames)

if not hasattr(measurement, "steadyState"):
    raise KeyError(f"No 'steadyState' group in {mat_files[0].name}")
example_records = np.atleast_1d(measurement.steadyState)
if len(example_records) == 0:
    raise ValueError(f"No steady-state records in {mat_files[0].name}")

example_record = example_records[0]
print("Fields in first steady-state record:", example_record._fieldnames)
print("Fields in its timeDomain structure:", example_record.timeDomain._fieldnames)

: 

## 3. Load one row per steady-state record

We iterate over the actual records and their inspected `timeDomain` fields. The extra `device_folder`, `source_file`, and `record_index` columns preserve provenance. `record_index` is only a row position within one file—it is **not** an aging-cycle count. Missing sensor values remain `NaN`; nothing is imputed or removed.

In [ ]:
import pandas as pd

rows = []
expected_signal_fields = list(example_record.timeDomain._fieldnames)

for mat_path in mat_files:
    contents = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    if "measurement" not in contents or not hasattr(contents["measurement"], "steadyState"):
        raise KeyError(f"Expected measurement.steadyState in {mat_path}")

    records = np.atleast_1d(contents["measurement"].steadyState)
    for record_index, record in enumerate(records):
        signal_fields = list(record.timeDomain._fieldnames)
        if signal_fields != expected_signal_fields:
            raise ValueError(
                f"Unexpected timeDomain fields in {mat_path.name}, record {record_index}: "
                f"{signal_fields}"
            )

        row = {
            "device_folder": mat_path.parent.name,
            "source_file": str(mat_path.relative_to(aging_folder)),
            "record_index": record_index,
            "date": record.date,
            "timeEpoch": float(np.asarray(record.timeEpoch).item()),
        }
        for field in signal_fields:
            value = np.asarray(getattr(record.timeDomain, field))
            if value.size != 1 or not np.issubdtype(value.dtype, np.number):
                raise TypeError(
                    f"Expected a numeric scalar for {field} in {mat_path.name}, "
                    f"record {record_index}; got shape={value.shape}, dtype={value.dtype}."
                )
            row[field] = float(value.item())
        rows.append(row)

df = pd.DataFrame(rows)
print(f"Loaded {len(df):,} steady-state records from {len(mat_files)} MAT files.")

## 4. Inspect rows, table dimensions, and data types

`df.head()` shows the first rows; it is useful for checking column names and spotting things like an empty-looking first sensor record. `df.shape` is `(rows, columns)`. `df.info()` shows each column's type and non-missing count. Pandas `object` or `string` columns here describe identifiers and recorded date text; the measurement channels are numeric floating-point values. A numeric row index is not the same thing as physical aging cycles.

In [ ]:
df.head()

: 

In [ ]:
df.shape

In [ ]:
df.info()

## 5. Check missing values and numerical ranges

First count missing values by column and by source file. `describe()` summarizes numeric columns (count, mean, standard deviation, quartiles, minimum, and maximum). These summaries are for inspection only: unusual values or ranges should not be corrected until measurement units, calibration, and experiment conditions are understood.

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
missing_by_column[missing_by_column > 0]

In [ ]:
missing_by_file = df.groupby("source_file", observed=True)[expected_signal_fields].apply(
    lambda group: group.isna().sum().sum()
)
missing_by_file.sort_values(ascending=False)

In [ ]:
df.describe()

In [ ]:
print("Columns by pandas dtype:")
print(df.dtypes.to_string())

numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
print("\nNumeric columns:", numeric_columns)
print("Identifier / text columns:", categorical_columns)

### How to interpret the initial missing-value and range results

On this local extraction, loading all ten `Aging Data` steady-state MAT files produces **68,769 rows and 13 columns**. Eight records have missing values in each of the seven populated `timeDomain` channels; `ambientTemperature` is missing in all 68,769 records. This is a property to report and investigate—not a reason to silently fill values or drop the channel. The first records with missing signals also have an implausible placeholder-looking date / zero `timeEpoch`; preserve them for now and decide how to handle them only in the cleaning phase.

Some numeric ranges appear surprising (for example, temperature-named fields). The dataset README warns that steady-state measurements are not properly scaled. **A field name is not proof of engineering units or calibration.** Keep the recorded numerical values unchanged and consult the experiment notes before interpreting them physically.

## 6. Inspect record timing and sampling information

`timeEpoch` is present in the steady-state records, but a timestamp is not a cycle number. Estimate sampling intervals separately inside each source file because files are separate segments and their clocks may restart. We use only positive, finite epoch values for this interval summary; the original values in `df` remain untouched. The median interval describes typical cadence, while the largest interval helps reveal pauses or acquisition gaps.

In [ ]:
sampling_rows = []
for source_file, group in df.groupby("source_file", sort=True, observed=True):
    timestamps = pd.to_numeric(group["timeEpoch"], errors="coerce")
    timestamps = timestamps[np.isfinite(timestamps) & (timestamps > 0)]
    intervals = timestamps.diff().dropna()
    positive_intervals = intervals[intervals > 0]

    sampling_rows.append({
        "source_file": source_file,
        "valid_timestamps": len(timestamps),
        "nonpositive_intervals": int((intervals <= 0).sum()),
        "median_interval_seconds": positive_intervals.median(),
        "minimum_interval_seconds": positive_intervals.min(),
        "maximum_interval_seconds": positive_intervals.max(),
    })

sampling_summary = pd.DataFrame(sampling_rows).set_index("source_file")
sampling_summary

In this local copy, valid steady-state timestamps have a typical interval around **0.2 seconds** (roughly five recorded steady-state observations per second), but intervals vary and some files contain much larger gaps. This is the cadence of the saved steady-state records—not the oscilloscope's sampling rate for the separate transient waveforms. A timestamp cadence also does not establish how many thermal cycles a device experienced.

## 7. Save the loaded table outside the raw-data folder

This CSV is a convenient, processed copy of the scalar `steadyState` records, with file and device provenance retained. It is **not** a lossless substitute for the nested MATLAB files, and it does not include transient waveforms. The raw MAT files are never modified.

In [ ]:
processed_folder = project_root / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)
processed_path = processed_folder / "aging_steady_state_observations.csv"

df.to_csv(processed_path, index=False)
print(f"Saved {len(df):,} rows to {processed_path.relative_to(project_root)}")

## Phase 2 learning check

Before moving on, answer in your own words:

1. Why does each row in this table represent one saved steady-state record, not one device or one thermal cycle?
2. Which columns are identifiers/date text, which are numeric record metadata, and which are numeric measurement channels? Does a numeric dtype guarantee a correct unit?
3. How many measurements are missing in `ambientTemperature`, and what does that tell you about whether to use that field?
4. What does the approximately 0.2-second median interval describe? Why is it not the transient oscilloscope sampling rate or an aging-cycle count?
5. Why should the main, `b`, and `check` files stay traceable as separate sources until their experimental relationship is established?

Send me your answers and any unexpected output. The next step will be data understanding / cleaning decisions—not feature extraction or RUL modeling.